# Preliminari

Si impostano directory di lavoro e si fanno import per spark

In [1]:
import os
from pyspark.sql import SparkSession

DATASETS_DIR = "../dataset/"

spark = (
    SparkSession.builder
    .appName("pfp")
    .getOrCreate()
)

sc = spark.sparkContext


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/05 16:43:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Pre-Processing


Creo un dataframe dal file .parquet di input.

Creo i record (rdd) come necessario dal problema, ovvero chiave dell'ordine e valore le tuple contenente id oggetto e quantità.
Questi record li chiameremo Transazioni, come suggerito dal paper PFP.

In [2]:
sdf = spark.read.parquet(
    os.path.join(DATASETS_DIR, "online_retail.parquet")
)

transactions = (
    sdf
    .select("InvoiceNo", "StockCode", "Quantity")
    .rdd
    .map(lambda row: (row["InvoiceNo"], (row["StockCode"], row["Quantity"])))
    .groupByKey()
    .mapValues(list)
)
transactions.take(5)

[('536365',
  [('85123A', '6'),
   ('71053', '6'),
   ('84406B', '8'),
   ('84029G', '6'),
   ('84029E', '6'),
   ('22752', '2'),
   ('21730', '6')]),
 ('536366', [('22633', '6'), ('22632', '6')]),
 ('536367',
  [('84879', '32'),
   ('22745', '6'),
   ('22748', '6'),
   ('22749', '8'),
   ('22310', '6'),
   ('84969', '6'),
   ('22623', '3'),
   ('22622', '2'),
   ('21754', '3'),
   ('21755', '3'),
   ('21777', '4'),
   ('48187', '4')]),
 ('536368', [('22960', '6'), ('22913', '3'), ('22912', '3'), ('22914', '3')]),
 ('536369', [('21756', '3')])]

### Conversione

convertiamo gli oggetti (tuple chiave e quantità) in nuovi oggetti identificati da un numero, in questo modo si può facilmente utilizzare PFP con le quantità.
Riduciamo funzionalmente il problema di tenere in considerazione le quantità al problema senza le quantità per poi tornare al problema delle quantità.

T' = T
for t in T'
    t -> t'

out = PFP(T')

reversed = revert(out)

return alpha_code(reversed)

Per fare tutto ciò innanzitutto devo prendere gli oggetti e quantità e mapparli:

In [14]:
pairs= (
    transactions
    .flatMap(lambda x: x[1])                 # prendo tutte le tuple
    .distinct()                              # tuple uniche
    .sortBy(lambda pair: (pair[0], pair[1])) # ordine stabile
    .zipWithIndex()                          # assegna indice 0,1,2...
)

print("ci sono " + str(pairs.count()) + " coppie")
pairs.take(50)


ci sono 45280 coppie


[(('10002', '-3'), 0),
 (('10002', '1'), 1),
 (('10002', '10'), 2),
 (('10002', '11'), 3),
 (('10002', '12'), 4),
 (('10002', '120'), 5),
 (('10002', '14'), 6),
 (('10002', '18'), 7),
 (('10002', '180'), 8),
 (('10002', '2'), 9),
 (('10002', '24'), 10),
 (('10002', '3'), 11),
 (('10002', '36'), 12),
 (('10002', '4'), 13),
 (('10002', '48'), 14),
 (('10002', '5'), 15),
 (('10002', '6'), 16),
 (('10002', '60'), 17),
 (('10002', '62'), 18),
 (('10002', '8'), 19),
 (('10080', '1'), 20),
 (('10080', '12'), 21),
 (('10080', '170'), 22),
 (('10080', '2'), 23),
 (('10080', '22'), 24),
 (('10080', '24'), 25),
 (('10080', '26'), 26),
 (('10080', '3'), 27),
 (('10080', '4'), 28),
 (('10080', '48'), 29),
 (('10120', '1'), 30),
 (('10120', '10'), 31),
 (('10120', '11'), 32),
 (('10120', '12'), 33),
 (('10120', '2'), 34),
 (('10120', '20'), 35),
 (('10120', '3'), 36),
 (('10120', '30'), 37),
 (('10120', '4'), 38),
 (('10120', '5'), 39),
 (('10120', '6'), 40),
 (('10120', '8'), 41),
 (('10123C', '-18

In [4]:
# Creo la mappa di conversione da tupla a numero
conversion_map = pairs.collectAsMap()
# Lo distribuisco ai worker in broadcast
bc_map = sc.broadcast(conversion_map)
bc_map.value

{('10002', '-3'): 1,
 ('10002', '1'): 2,
 ('10002', '10'): 3,
 ('10002', '11'): 4,
 ('10002', '12'): 5,
 ('10002', '120'): 6,
 ('10002', '14'): 7,
 ('10002', '18'): 8,
 ('10002', '180'): 9,
 ('10002', '2'): 10,
 ('10002', '24'): 11,
 ('10002', '3'): 12,
 ('10002', '36'): 13,
 ('10002', '4'): 14,
 ('10002', '48'): 15,
 ('10002', '5'): 16,
 ('10002', '6'): 17,
 ('10002', '60'): 18,
 ('10002', '62'): 19,
 ('10002', '8'): 20,
 ('10080', '1'): 21,
 ('10080', '12'): 22,
 ('10080', '170'): 23,
 ('10080', '2'): 24,
 ('10080', '22'): 25,
 ('10080', '24'): 26,
 ('10080', '26'): 27,
 ('10080', '3'): 28,
 ('10080', '4'): 29,
 ('10080', '48'): 30,
 ('10120', '1'): 31,
 ('10120', '10'): 32,
 ('10120', '11'): 33,
 ('10120', '12'): 34,
 ('10120', '2'): 35,
 ('10120', '20'): 36,
 ('10120', '3'): 37,
 ('10120', '30'): 38,
 ('10120', '4'): 39,
 ('10120', '5'): 40,
 ('10120', '6'): 41,
 ('10120', '8'): 42,
 ('10123C', '-18'): 43,
 ('10123C', '1'): 44,
 ('10123C', '3'): 45,
 ('10123G', '-38'): 46,
 ('10124

In [5]:
# Creo la mappa di conversione da tupla a numero
conversion_map = pairs.collectAsMap()
# Lo distribuisco ai worker in broadcast
bc_map = sc.broadcast(conversion_map)

# Rimappo ogni transazione
transactions_ids = transactions.mapValues(
    lambda items: [bc_map.value[item] for item in items]
)
print(transactions_ids.count())
transactions_ids.take(5)

25900


[('536365', [42871, 36544, 38857, 38275, 38242, 23017, 9285]),
 ('536366', [21178, 21148]),
 ('536367',
  [40566,
   22914,
   22959,
   22976,
   16064,
   41171,
   20968,
   20952,
   9514,
   9531,
   9632,
   36265]),
 ('536368', [25907, 25162, 25152, 25172]),
 ('536369', [9544])]

## PFP

Iniziamo ad implementare **PFP**, definiamo una variabile **epsilon** che rappresenta la "predefined minimum support threshold"
soglia minima predefinita di supporto.
Quindi una threshold sopra la quale verrà riconosciuto un pattern e i pattern sotto questa soglia verranno scartati 

In [6]:
epsilon = 100

item_counts = (
    transactions_ids
    .flatMap(lambda row: set(row[1]))   # ogni item contato una sola volta per transazione
    .map(lambda item: (item, 1))
    .reduceByKey(lambda a, b: a + b)
    .filter(lambda row: row[1] >= epsilon)
)
print(item_counts.count())
item_counts.take(10)


970


[(23017, 121),
 (42871, 560),
 (22914, 138),
 (9514, 269),
 (22959, 138),
 (20952, 121),
 (9531, 201),
 (25907, 451),
 (9544, 103),
 (45259, 197)]

Creo la F-List che è la lista decrescente degli item (item = id_of(tuple(code, quantity)))

Poi ordino le transazione per "supporto" ovvero in base ai valori di F-List

Creo infine la Q-List tramite la quale si suddivide il calcolo tra le macchine.

In [7]:
# creo f_list
f_list = item_counts.sortBy(
    lambda row: (row[1], row[0]),
    ascending=False
)

f_list.take(10)

[(42544, 724),
 (45186, 708),
 (1995, 631),
 (40578, 597),
 (17797, 596),
 (5318, 583),
 (22437, 571),
 (42871, 560),
 (29326, 551),
 (29509, 513)]

In [8]:
# Creo una mappa per ordinare le transazioni, la mappa è fatta così:  item_id -> posizione nella F-list

# Base comune: item_id -> rank nella F-list
item_rank = (
    f_list
    .map(lambda row: row[0])      # item_id
    .zipWithIndex()               # item_id -> rank
    .map(lambda x: (x[0], int(x[1])))
    .persist()
)

f_rank = item_rank.collectAsMap()
# notifico i worker
bc_f_rank = sc.broadcast(f_rank)

ordered_transactions = (
    transactions_ids
    .mapValues(
        lambda items: sorted(
            # tieni item solo se item è una chiave del dizionario f_rank
            set(item for item in items if item in bc_f_rank.value),
            key=lambda item: bc_f_rank.value[item]
        )
    )
    .filter(lambda row: len(row[1]) > 0)
)
print(ordered_transactions.count())
print(ordered_transactions.take(5))


16578
[('536365', [42871, 23017]), ('536367', [9514, 9531, 22959, 22914, 20952]), ('536368', [25907]), ('536369', [9544]), ('536370', [18813, 45259])]


In [9]:
# G-List
Q = spark.sparkContext.defaultParallelism # numero di core disponibili tra tutti i worker

g_list = (
    item_rank
    .map(lambda x: (x[0], int(x[1] % Q)))   # item_id -> gid
    .collectAsMap()
)

bc_g_list = sc.broadcast(g_list)

In [10]:
#Questo è fondamentalmente il mapper del paper
def generate_group_dependent_transactions(row):
    invoice_no, items = row

    output = []
    seen_gids = set()

    # Scorro la transazione da destra verso sinistra
    for j in range(len(items) - 1, -1, -1):
        item = items[j]
        gid = bc_g_list.value.get(item)

        # Se questo gruppo non è ancora stato emesso per questa transazione
        if gid is not None and gid not in seen_gids:
            seen_gids.add(gid)

            # Emetto il prefisso fino alla posizione j inclusa
            output.append((gid, items[:j + 1]))

    return output

group_dependent_transactions = ordered_transactions.flatMap(
    generate_group_dependent_transactions
)

group_dependent_transactions.take(10)

# a ogni gid associo le transazioni di cui si deve occupare.

group_shards = group_dependent_transactions.groupByKey()

### Nodi e Alberi

A questo punto abbiamo bisogno degli FP-tree per minare i pattern. 
Ogni worker/reducer costruirà un FP-tree locale a partire dalle transazioni associate a uno specifico gruppo.

Dato che gli item di ogni transazione sono già ordinati secondo la F-list, e dato che le transazioni group-dependent sono raggruppate per `gid`, ogni gruppo può essere minato in modo indipendente dagli altri.

Una transazione originale può generare più transazioni parziali, una per ogni gruppo presente nella transazione. Durante lo shuffle, questi prefissi vengono inviati ai reducer corrispondenti ai rispettivi gruppi.

Quindi una stessa transazione originale può contribuire a più FP-tree locali, ma ogni FP-tree locale contiene solo il sotto-database necessario per minare i pattern che terminano negli item del proprio gruppo.

In [11]:

from collections import defaultdict

# Creiamo una struttura di nodi in grado di navigare al parent e ai child.
class FPNode:
    def __init__(self, item=None, parent=None):
        self.item = item
        self.count = 0
        self.parent = parent
        self.children = {}

    def add_child(self, item):
        child = FPNode(item=item, parent=self)
        self.children[item] = child
        return child


# L'inserimento di una transazione può comportare la creazione di nodi figli oppure l'incremento del loro conteggio.

def insert_transaction(root, transaction, header_table, count=1):
    node = root

    for item in transaction:
        if item in node.children:
            child = node.children[item]
            child.count += count
        else:
            child = node.add_child(item)
            child.count = count
            header_table[item].append(child)

        node = child
# La header-table serve per avere una navigazione rapida ai nodi che rappresentano lo stesso item, infatti nell'albero possono 
# esserci N nodi che rappresentano lo stesso item, grazie alla header_table possiamo evitare di navigare tutto l'albero 
# ma abbiamo un accesso "diretto"

def build_fp_tree(transactions_iter):
    root = FPNode()
    header_table = defaultdict(list)

    for transaction in transactions_iter:
        insert_transaction(root, transaction, header_table, count=1)

    return root, dict(header_table)


def print_tree(node, indent=0, max_depth=3):
    if indent >= max_depth:
        return

    for child in node.children.values():
        print("  " * indent + f"{child.item}:{child.count}")
        print_tree(child, indent + 1, max_depth)



def count_nodes(node):
    total = 1
    for child in node.children.values():
        total += count_nodes(child)
    return total



def tree_stats_for_group(row):
    gid, transactions_iter = row

    root, header_table = build_fp_tree(transactions_iter)

    return (
        gid,
        count_nodes(root),
        len(header_table),
        list(root.children.keys())[:10]
    )

In [12]:
# Ora creiamo la nowGroup

gid_to_items_tmp = defaultdict(list)

for item_id, gid in g_list.items():
    gid_to_items_tmp[gid].append(item_id)
# gid -> item_ids
nowGroup = dict(gid_to_items_tmp)

bc_nowGroup = sc.broadcast(nowGroup)

In [13]:
tree_stats = group_shards.map(tree_stats_for_group)
gid, nodes_number, header_table_length,childern_length = tree_stats.take(1)[0]
tree_stats.take(10)
print("gid:", gid)
print("item distinti nell'albero:", header_table_length)

gid: 6
item distinti nell'albero: 948
